In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

from tqdm.auto import tqdm

c:\Users\Dell\Documents\Projects\AgriSense\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Reproducability + device

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [3]:
# project paths

project_root = Path.cwd().parent

if not (project_root / "data").exists():
    project_root = Path.cwd().parent.parent

print("Project root:", project_root)

Project root: c:\Users\Dell\Documents\Projects\AgriSense


In [4]:
#loading clean metedata
processed_dir = project_root / "data" / "processed"

clean_metadata_path = (
    processed_dir / "agrisense_metadata_clean.csv"
)

df_clean = pd.read_csv(clean_metadata_path)

print("Shape:", df_clean.shape)
print("Columns:", df_clean.columns.tolist())

Shape: (7399, 11)
Columns: ['Name', 'Index', 'Plant', 'Disease', 'Resolution', 'Label file', 'Mask ratio', 'URL', 'License', 'Split', 'image_hash']


In [5]:
#disease mapping

disease_classes = sorted(df_clean["Disease"].unique())

disease_to_id = {
    disease: idx
    for idx, disease in enumerate(disease_classes)
}

id_to_disease = {
    idx: disease
    for disease, idx in disease_to_id.items()
}

df_clean["disease_id"] = df_clean["Disease"].map(disease_to_id)

num_classes = len(disease_classes)

print("Disease classes:", num_classes)
print("Missing disease IDs:", df_clean["disease_id"].isna().sum())

Disease classes:

 115
Missing disease IDs: 0


In [6]:
# resolve actual image paths

image_root = (
    project_root
    / "data"
    / "raw"
    / "PlantSeg"
    / "images"
)

def find_actual_image_path(row):
    expected_path = image_root / row["Name"]

    if expected_path.exists():
        return expected_path

    matches = list(image_root.rglob(row["Name"]))

    if len(matches) == 1:
        return matches[0]

    return None


df_clean["actual_image_path"] = df_clean.apply(
    find_actual_image_path,
    axis=1
)

print(
    "Missing image paths:",
    df_clean["actual_image_path"].isna().sum()
)

Missing image paths: 0


In [7]:
# Splitting the Dataset

train_df = df_clean[df_clean["Split"] == "Training"].reset_index(drop=True)
val_df = df_clean[df_clean["Split"] == "Validation"].reset_index(drop=True)
test_df = df_clean[df_clean["Split"] == "Test"].reset_index(drop=True)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Training: 5084
Validation: 811
Test: 1504


In [8]:
# Preprocessing (Same as baseline)

class ResizeWithPadding:
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        target_h, target_w = self.size

        img.thumbnail((target_w, target_h), Image.Resampling.LANCZOS)

        canvas = Image.new(
            "RGB",
            (target_w, target_h),
            (0, 0, 0)
        )

        left = (target_w - img.width) // 2
        top = (target_h - img.height) // 2

        canvas.paste(img, (left, top))

        return canvas

In [9]:
train_transform = transforms.Compose([
    ResizeWithPadding((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    ResizeWithPadding((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [10]:
#Dataset

class AgriSenseDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = Image.open(
            row["actual_image_path"]
        ).convert("RGB")

        label = int(row["disease_id"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [11]:
#DataLoaders

BATCH_SIZE = 16

train_dataset = AgriSenseDataset(
    train_df,
    transform=train_transform
)

val_dataset = AgriSenseDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = AgriSenseDataset(
    test_df,
    transform=val_test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 318
Validation batches: 51
Test batches: 94


In [12]:
# Creating a fine tuning model

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

# Freeze entire backbone first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze Layer 4
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace classification head
model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)

In [13]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

Trainable parameters: 8452723
Total parameters: 11235507


In [14]:
# loss + Optimizer

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

In [15]:
# Training + Evaluation Functions

def run_epoch(model, loader, criterion, optimizer=None):

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    progress_bar = tqdm(
        loader,
        leave=False
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            if is_training:
                loss.backward()
                optimizer.step()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        running_loss += (
            loss.item() * images.size(0)
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        all_targets.extend(
            labels.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss / len(loader.dataset)
    )

    epoch_accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_accuracy,
        epoch_macro_f1
    )

In [19]:
# Training Loop

NUM_EPOCHS = 5

checkpoint_dir = (
    project_root
    / "models"
    / "checkpoints"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

latest_checkpoint_path = (
    checkpoint_dir
    / "resnet18_finetuned_latest.pth"
)

best_checkpoint_path = (
    checkpoint_dir
    / "resnet18_finetuned_best.pth"
)

best_val_macro_f1 = -1

history = []

for epoch in range(1, NUM_EPOCHS + 1):

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss, train_acc, train_f1 = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc, val_f1 = run_epoch(
        model,
        val_loader,
        criterion
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1
    })

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Train Macro F1: {train_f1:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1
    }

    torch.save(
        checkpoint,
        latest_checkpoint_path
    )

    if val_f1 > best_val_macro_f1:

        best_val_macro_f1 = val_f1

        torch.save(
            checkpoint,
            best_checkpoint_path
        )

        print("Best checkpoint updated.")



Epoch 1/5


Train Loss: 3.3956 | Train Acc: 0.2555 | Train Macro F1: 0.1191
Val Loss: 2.4812 | Val Acc: 0.4402 | Val Macro F1: 0.2410
Best checkpoint updated.

Epoch 2/5


Train Loss: 2.1564 | Train Acc: 0.4970 | Train Macro F1: 0.3191
Val Loss: 1.9676 | Val Acc: 0.4994 | Val Macro F1: 0.3138
Best checkpoint updated.

Epoch 3/5


Train Loss: 1.6015 | Train Acc: 0.6231 | Train Macro F1: 0.4752
Val Loss: 1.7127 | Val Acc: 0.5598 | Val Macro F1: 0.3854
Best checkpoint updated.

Epoch 4/5


Train Loss: 1.2306 | Train Acc: 0.7118 | Train Macro F1: 0.5944
Val Loss: 1.6097 | Val Acc: 0.5610 | Val Macro F1: 0.4008
Best checkpoint updated.

Epoch 5/5


Train Loss: 0.9462 | Train Acc: 0.7884 | Train Macro F1: 0.6913
Val Loss: 1.5561 | Val Acc: 0.5684 | Val Macro F1: 0.4177
Best checkpoint updated.


In [17]:
project_root = Path.cwd().parent

if not (project_root / "data").exists():
    project_root = Path.cwd().parent.parent

checkpoint_dir = (
    project_root
    / "models"
    / "checkpoints"
)

best_checkpoint_path = (
    checkpoint_dir
    / "resnet18_finetuned_best.pth"
)

print("Checkpoint path:", best_checkpoint_path)
print("Checkpoint exists:", best_checkpoint_path.exists())

Checkpoint path: c:\Users\Dell\Documents\Projects\AgriSense\models\checkpoints\resnet18_finetuned_best.pth
Checkpoint exists: True


In [18]:
# Load the best Checkpoint

best_checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.eval()

print("Best checkpoint epoch:", best_checkpoint["epoch"])
print("Best validation loss:", best_checkpoint["val_loss"])
print("Best validation accuracy:", best_checkpoint["val_accuracy"])
print("Best validation Macro F1:", best_checkpoint["val_macro_f1"])

Best checkpoint epoch: 5
Best validation loss: 1.5560793709078788
Best validation accuracy: 0.5684340320591862
Best validation Macro F1: 0.4176858941901695


In [19]:
# Test Preditions

test_predictions_finetuned = []
test_targets_finetuned = []

with torch.no_grad():

    test_bar = tqdm(
        test_loader,
        desc="Testing fine-tuned model"
    )

    for images, labels in test_bar:

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_predictions_finetuned.extend(
            predictions.cpu().numpy()
        )

        test_targets_finetuned.extend(
            labels.numpy()
        )

print("Test samples:", len(test_targets_finetuned))
print("Predictions:", len(test_predictions_finetuned))

Testing fine-tuned model: 100%|██████████| 94/94 [01:31<00:00,  1.03it/s]

Test samples: 1504
Predictions: 1504


In [20]:
#Test Metrices

finetuned_accuracy = accuracy_score(
    test_targets_finetuned,
    test_predictions_finetuned
)

finetuned_macro_f1 = f1_score(
    test_targets_finetuned,
    test_predictions_finetuned,
    average="macro",
    zero_division=0
)

finetuned_weighted_f1 = f1_score(
    test_targets_finetuned,
    test_predictions_finetuned,
    average="weighted",
    zero_division=0
)

print(f"Fine-tuned Test Accuracy: {finetuned_accuracy:.4f}")
print(f"Fine-tuned Test Macro F1: {finetuned_macro_f1:.4f}")
print(f"Fine-tuned Test Weighted F1: {finetuned_weighted_f1:.4f}")

Fine-tuned Test Accuracy: 0.5944
Fine-tuned Test Macro F1: 0.4650
Fine-tuned Test Weighted F1: 0.5779


In [21]:
#Fine Tuned classification report

finetuned_report = classification_report(
    test_targets_finetuned,
    test_predictions_finetuned,
    labels=list(range(num_classes)),
    target_names=[
        id_to_disease[i]
        for i in range(num_classes)
    ],
    output_dict=True,
    zero_division=0
)

finetuned_report_df = (
    pd.DataFrame(finetuned_report)
    .T
    .iloc[:num_classes]
    .copy()
)

finetuned_report_df.index.name = "Disease"
finetuned_report_df = finetuned_report_df.reset_index()

display(
    finetuned_report_df[
        ["Disease", "f1-score", "support"]
    ].sort_values(
        "f1-score",
        ascending=False
    ).head(15)
)

,Disease,f1-score,support
30,carrot cavity spot,1.000000,9.0
5,banana black leaf streak,0.893617,24.0
6,banana bunchy top,0.888889,13.0
109,wheat stem rust,0.878049,22.0
71,plum brown rot,0.875000,8.0
27,cabbage black rot,0.864865,19.0
52,eggplant phomopsis fruit rot,0.857143,7.0
15,bell pepper blossom end rot,0.848485,16.0
9,banana panama disease,0.842105,10.0
47,corn smut,0.842105,26.0


In [23]:
# Same baeline code as previous for folowing cells

# Recreate baseline model
baseline_model = models.resnet18(weights=None)

baseline_model.fc = nn.Linear(
    baseline_model.fc.in_features,
    num_classes
)

baseline_model = baseline_model.to(device)

baseline_checkpoint_path = (
    project_root
    / "models"
    / "checkpoints"
    / "resnet18_baseline_best.pth"
)

baseline_checkpoint = torch.load(
    baseline_checkpoint_path,
    map_location=device
)

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"]
)

baseline_model.eval()

print("Baseline checkpoint epoch:", baseline_checkpoint["epoch"])
print("Baseline validation loss:", baseline_checkpoint["val_loss"])
print("Baseline validation accuracy:", baseline_checkpoint["val_accuracy"])
print("Baseline validation Macro F1:", baseline_checkpoint["val_macro_f1"])

Baseline checkpoint epoch: 5
Baseline validation loss: 2.2170482657840602
Baseline validation accuracy: 0.42293464858199753
Baseline validation Macro F1: 0.29562832708362424


In [24]:
# Generating baseline Test Predictions

test_predictions_baseline = []
test_targets_baseline = []

with torch.no_grad():

    test_bar = tqdm(
        test_loader,
        desc="Testing baseline model"
    )

    for images, labels in test_bar:

        images = images.to(device)

        outputs = baseline_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_predictions_baseline.extend(
            predictions.cpu().numpy()
        )

        test_targets_baseline.extend(
            labels.numpy()
        )

print("Baseline test samples:", len(test_targets_baseline))
print("Baseline predictions:", len(test_predictions_baseline))

Testing baseline model: 100%|██████████| 94/94 [01:30<00:00,  1.04it/s]

Baseline test samples: 1504
Baseline predictions: 1504


In [25]:
# Creating baseline classification report

baseline_report = classification_report(
    test_targets_baseline,
    test_predictions_baseline,
    labels=list(range(num_classes)),
    target_names=[
        id_to_disease[i]
        for i in range(num_classes)
    ],
    output_dict=True,
    zero_division=0
)

baseline_report_df = (
    pd.DataFrame(baseline_report)
    .T
    .iloc[:num_classes]
    .copy()
)

baseline_report_df.index.name = "Disease"
baseline_report_df = baseline_report_df.reset_index()

print("Baseline disease report created.")

display(
    baseline_report_df[
        ["Disease", "f1-score", "support"]
    ].head()
)

Baseline disease report created.


,Disease,f1-score,support
0,apple black rot,0.333333,12.0
1,apple mosaic virus,0.222222,13.0
2,apple rust,0.000000,22.0
3,apple scab,0.461538,35.0
4,banana anthracnose,0.518519,9.0


In [26]:
# Creating finetuned report

finetuned_report = classification_report(
    test_targets_finetuned,
    test_predictions_finetuned,
    labels=list(range(num_classes)),
    target_names=[
        id_to_disease[i]
        for i in range(num_classes)
    ],
    output_dict=True,
    zero_division=0
)

finetuned_report_df = (
    pd.DataFrame(finetuned_report)
    .T
    .iloc[:num_classes]
    .copy()
)

finetuned_report_df.index.name = "Disease"
finetuned_report_df = finetuned_report_df.reset_index()

print("Fine-tuned disease report created.")

Fine-tuned disease report created.


In [27]:
#comparing baseline vs fine Tuning

finetuned_comparison_df = baseline_report_df[
    ["Disease", "f1-score", "support"]
].rename(
    columns={"f1-score": "Baseline F1"}
)

finetuned_comparison_df["Fine-tuned F1"] = (
    finetuned_report_df["f1-score"]
)

finetuned_comparison_df["F1 Change"] = (
    finetuned_comparison_df["Fine-tuned F1"]
    - finetuned_comparison_df["Baseline F1"]
)

finetuned_comparison_df = (
    finetuned_comparison_df
    .sort_values("F1 Change", ascending=False)
)

display(
    finetuned_comparison_df.head(15)
)

display(
    finetuned_comparison_df.tail(15)
)

,Disease,Baseline F1,support,Fine-tuned F1,F1 Change
77,raspberry fire blight,0.000000,4.0,0.500000,0.500000
55,garlic rust,0.200000,18.0,0.689655,0.489655
71,plum brown rot,0.400000,8.0,0.875000,0.475000
80,raspberry yellow rust,0.285714,5.0,0.750000,0.464286
109,wheat stem rust,0.413793,22.0,0.878049,0.464256
2,apple rust,0.000000,22.0,0.454545,0.454545
52,eggplant phomopsis fruit rot,0.444444,7.0,0.857143,0.412698
91,strawberry leaf scorch,0.000000,3.0,0.400000,0.400000
76,potato late blight,0.292683,18.0,0.666667,0.373984
69,peach scab,0.470588,11.0,0.818182,0.347594


,Disease,Baseline F1,support,Fine-tuned F1,F1 Change
112,zucchini downy mildew,0.000000,5.0,0.000000,0.000000
74,plum rust,0.000000,4.0,0.000000,0.000000
79,raspberry leaf spot,0.000000,1.0,0.000000,0.000000
70,plum bacterial spot,0.000000,3.0,0.000000,0.000000
72,plum pocket disease,0.000000,4.0,0.000000,0.000000
56,ginger leaf spot,0.285714,5.0,0.250000,-0.035714
73,plum pox virus,0.500000,5.0,0.444444,-0.055556
18,blueberry anthracnose,0.666667,4.0,0.600000,-0.066667
17,bell pepper powdery mildew,0.400000,4.0,0.333333,-0.066667
78,raspberry gray mold,0.666667,2.0,0.500000,-0.166667


In [28]:
#Lets Count improvement

improved = (
    finetuned_comparison_df["F1 Change"] > 0
).sum()

unchanged = (
    finetuned_comparison_df["F1 Change"] == 0
).sum()

declined = (
    finetuned_comparison_df["F1 Change"] < 0
).sum()

print("Diseases improved:", improved)
print("Diseases unchanged:", unchanged)
print("Diseases declined:", declined)

Diseases improved: 82
Diseases unchanged: 23
Diseases declined: 10


In [29]:
# Repeating support Group Analysis Similar to Experiment 1

finetuned_comparison_df["Support Group"] = pd.cut(
    finetuned_comparison_df["support"],
    bins=[0, 5, 10, 20, 50, float("inf")],
    labels=["1-5", "6-10", "11-20", "21-50", "51+"]
)

finetuned_support_summary = (
    finetuned_comparison_df
    .groupby("Support Group", observed=True)
    .agg(
        Diseases=("Disease", "count"),
        Mean_Baseline_F1=("Baseline F1", "mean"),
        Mean_Finetuned_F1=("Fine-tuned F1", "mean"),
        Mean_F1_Change=("F1 Change", "mean")
    )
    .reset_index()
)

display(finetuned_support_summary)

,Support Group,Diseases,Mean_Baseline_F1,Mean_Finetuned_F1,Mean_F1_Change
0,1-5,31,0.188453,0.241116,0.052663
1,6-10,27,0.363486,0.461782,0.098296
2,11-20,35,0.379513,0.541127,0.161614
3,21-50,20,0.484745,0.666048,0.181302
4,51+,1,0.687500,0.805369,0.117869


## Experiment 2 Conclusion — Fine-Tuning ResNet-18

Fine-tuning was evaluated to determine whether adapting pretrained ImageNet visual features to the PlantSeg disease classification task would improve performance.

The experiment kept the dataset, train/validation/test splits, preprocessing, batch size, and evaluation procedure consistent with the baseline. The ResNet-18 backbone was initially frozen, after which Layer 4 was unfrozen along with the final classification layer.

### Test Performance

| Metric | Baseline | Fine-Tuned | Change |
|---|---:|---:|---:|
| Accuracy | 0.4402 | 0.5944 | +0.1542 |
| Macro F1 | 0.3399 | 0.4650 | +0.1251 |
| Weighted F1 | 0.4237 | 0.5779 | +0.1542 |

Fine-tuning improved 82 of the 115 disease classes, while 23 remained unchanged and 10 declined.

The improvement was observed across all test-set support groups:

| Test Support | Diseases | Mean F1 Change |
|---|---:|---:|
| 1–5 | 31 | +0.0527 |
| 6–10 | 27 | +0.0983 |
| 11–20 | 35 | +0.1616 |
| 21–50 | 20 | +0.1813 |
| 51+ | 1 | +0.1179 |

The results indicate that adapting deeper pretrained features to the agricultural disease domain substantially improved classification performance, including for minority disease classes.

### Model Selection

The fine-tuned ResNet-18 model is retained as the current classification model for subsequent AgriSense development.

Checkpoint:

`models/checkpoints/resnet18_finetuned_best.pth`